<a href="https://colab.research.google.com/github/eelcofolkertsma/BIX-samples/blob/main/sign_message.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
from jsonschema import validate, ValidationError
from jsonschema.validators import RefResolver

# Define your JSON schema
schema={}
with open('/content/iata-baggage-bag-segment-instruction.v1.0.0-alpha.3.json', 'r') as f:
        schema = json.loads(f.read())

# Load the referenced library schema locally and add it to the resolver's store
# This is crucial because the main schema references it by its online $id,
# but we have it locally.
referenced_library_schema_path = '/content/iata-baggage-library.v1.0.0-alpha.3.json'
referenced_library_schema_uri = 'https://schemas.developer.iata.org/standards/psc/baggage/alpha/iata-baggage-library.v1.0.0-alpha.3.json'
library_schema_content = {}
try:
    with open(referenced_library_schema_path, 'r') as f:
        library_schema_content = json.loads(f.read())
    # IMPORTANT: Override the $id within the loaded library schema to match the URI
    # that the main schema uses to reference it. This prevents RefResolver KeyErrors.
    if '$id' in library_schema_content and library_schema_content['$id'] != referenced_library_schema_uri:
        print(f"Correcting library schema $id from {library_schema_content['$id']} to {referenced_library_schema_uri}")
        library_schema_content['$id'] = referenced_library_schema_uri
except FileNotFoundError:
    print(f"Error: Referenced library schema not found at {referenced_library_schema_path}")
    # Depending on requirements, you might want to raise an error or handle it differently
except json.JSONDecodeError as e:
    print(f"Error decoding referenced library schema: {e}")
    # Handle error or raise

# Create a resolver to handle external references.
# The base_uri should point to the directory where the referenced schemas are located.
# We pre-populate the store with our local copy of the referenced schema, ensuring the correct URI is used.
resolver = RefResolver(
    base_uri='file:///content/',
    referrer=schema, # This is the main schema that might have relative references
    store={referenced_library_schema_uri: library_schema_content}
)

# Function to validate data against a schema
def validate_json_file(file_content, schema, resolver):
    try:
        data = json.loads(file_content)
        validate(instance=data, schema=schema, resolver=resolver)
        print("Validation successful!")
        return True
    except json.JSONDecodeError as e:
        print(f"Invalid JSON format: {e}")
        return False
    except ValidationError as e:
        print(f"Validation failed: {e.message}")
        print(f"Path: {'/'.join(map(str, e.path))}")
        return False


/tmp/ipykernel_3259/2394200393.py:3: DeprecationWarning: jsonschema.RefResolver is deprecated as of v4.18.0, in favor of the https://github.com/python-jsonschema/referencing library, which provides more compliant referencing behavior as well as more flexible APIs for customization. A future release will remove RefResolver. Please file a feature request (on referencing) if you are missing an API for the kind of customization you need.
  from jsonschema.validators import RefResolver


FileNotFoundError: [Errno 2] No such file or directory: '/content/iata-baggage-bag-segment-instruction.v1.0.0-alpha.3.json'

In [ ]:
print("\n--- Validating data from '/content/01. Identify_bag_to_handler-BagRQ.json' ---")
try:
    with open('/content/01. Identify_bag_to_handler-BagRQ.json', 'r') as f:
        file_content_from_disk = f.read()
    validate_json_file(file_content_from_disk, schema, resolver)
except FileNotFoundError:
    print("Error: The file '/content/01. Identify_bag_to_handler-BagRQ.json' was not found.")
# The validate_json_file function now handles json.JSONDecodeError and ValidationError internally,
# so we don't need a generic 'except Exception' here.
# Any remaining unexpected errors will be raised normally.


--- Validating data from '/content/01. Identify_bag_to_handler-BagRQ.json' ---
Validation successful!
